In [0]:
import pandas as pd
from azure.identity import ClientSecretCredential
from azure.keyvault.secrets import SecretClient

filename = dbutils.widgets.get("filename")

print("Received file:", filename)

tenant_id = "9f30390c-d905-4714-becf-cff6ed274517"
client_id = "e272c73f-d6e7-4f84-8a23-467ee31f7244"
client_secret = "a909baba-151e-4ace-9b70-27df10f7274a"

credential = ClientSecretCredential(
    tenant_id,
    client_id,
    client_secret
)

# Key Vault name only (not full URL)
key_vault_name = "databricks89vault"

vault_url = f"https://{key_vault_name}.vault.azure.net"


# Read SAS token from Key Vault
sas_token = client.get_secret(
    "raw"
).value

storage_account = "adfstoragepoc89"

file_path = (
    f"{sas_token}"
)

pdf = pd.read_csv(file_path)
df = spark.createDataFrame(pdf)

display(df)

In [0]:
from pyspark.sql.functions import col

# Transformation
transformed_df = (
    df.filter(col("Amount") > 1000)
      .dropDuplicates()
)

display(transformed_df)

In [0]:
transformed_pdf = transformed_df.toPandas()

transformed_pdf.to_csv(
    "/tmp/processed_{filename}",
    index=False
)

print("Output file created")

In [0]:
from azure.storage.blob import BlobClient
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

# Convert Spark dataframe to pandas
transformed_pdf = transformed_df.toPandas()

# Save locally first
local_file = "/tmp/processed_{filename}"
transformed_pdf.to_csv(local_file, index=False)

# Key Vault name only (not full URL)
key_vault_name = "databricks89vault"

vault_url = f"https://{key_vault_name}.vault.azure.net"

# Authenticate
tenant_id = "9f30390c-d905-4714-becf-cff6ed274517"
client_id = "e272c73f-d6e7-4f84-8a23-467ee31f7244"
client_secret = "a909baba-151e-4ace-9b70-27df10f7274a"

credential = ClientSecretCredential(
    tenant_id,
    client_id,
    client_secret
)
# Read SAS token from Key Vault
sas_token = client.get_secret(
    "processed"
).value

storage_account = "adfstoragepoc89"

file_path = (
    f"{sas_token}"
)

# Upload to ADLS/Blob
blob = BlobClient.from_blob_url(
    f"{sas_token}"
)

with open(local_file, "rb") as data:
    blob.upload_blob(data, overwrite=True)

print("File uploaded successfully")